In [1]:
# !pip install openai pillow dotenv

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from openai import OpenAI
client = OpenAI()

In [4]:
import base64
from PIL import Image
import io

def image_to_data_url(path):
    # Open the image (any format Pillow supports)
    im = Image.open(path)

    # Convert to PNG in memory
    buffer = io.BytesIO()
    im.save(buffer, format="PNG")
    buffer.seek(0)
    image_bytes = buffer.read()

    # Encode as base64
    b64 = base64.b64encode(image_bytes).decode()

    # MIME type is always PNG now
    mime = "image/png"

    return f"data:{mime};base64,{b64}"

In [5]:
import glob

# --- Hardcoded few-shot CSV ---
c36_csv = """
ID;x-coord;y-coord;sloj P;;;;sloj 37;;;;sloj 38;;;;sloj 39;;;;sloj 40;;;;meziloží slojí 37-38;;;;;;;meziloží slojí 38-39;;;;;;;meziloží slojí 39-40;;;;;;
;;;mocnost sloje P (cm);dobývaná mocnost sloje P (cm);kota sloje P (m n.m.);hloubka sloje P pod povrchem (m);mocnost sloje 37 (cm);dobývaná mocnost sloje 37 (cm);kota sloje 37 (m n.m.);hloubka sloje 37 pod povrchem (m);mocnost sloje 38 (cm);dobývaná mocnost sloje 38 (cm);kota sloje 38 (m n.m.);hloubka sloje 38 pod povrchem (m);mocnost sloje 39 (cm);dobývaná mocnost sloje 39 (cm);kota sloje 39 (m n.m.);hloubka sloje 39 pod povrchem (m);mocnost sloje 40;dobývaná mocnost sloje 40;kota sloje 40;hloubka sloje 40 pod povrchem;mocnost meziloží 37-38 (m);celková mocnost pískovců a slepenců (m);počet lavic pískovců a slepencůl o mocnosti větší než 5 m;počet lavic pískovců a slepencůl o mocnosti větší než 10 m;max mocnost lavic pískovců a slepenců (m);% kompetentních hornin;redukovaná pevnost hornin efektivního nadloží sloje 38 (Mpa);mocnost meziloží 38-39 (m);celková mocnost pískovců a slepenců (m);počet lavic pískovců a slepencůl o mocnosti větší než 5 m;počet lavic pískovců a slepencůl o mocnosti větší než 10 m;max mocnost lavic pískovců a slepenců (m);% kompetentních hornin;redukovaná pevnost hornin efektivního nadloží sloje 39 (Mpa);mocnost meziloží 39-40 (m);celková mocnost pískovců a slepenců (m);počet lavic pískovců a slepencůl o mocnosti větší než 5 m;počet lavic pískovců a slepencůl o mocnosti větší než 10 m;max mocnost lavic pískovců a slepenců (m);% kompetentních hornin;redukovaná pevnost hornin efektivního nadloží sloje 39 (Mpa)
C 36-92;-458186,8;-1103139,5;x;x;x;x;x;x;x;x;350;;-389,6;639,6;517;;-425,7;675,7;637;;-487,52;737,5;x;x;x;x;x;x;x;29,43;20,25;1;1;19,55;69;x;55,45;50;4;2;21,2;90;x
"""

c37_csv = """
ID;x-coord;y-coord;sloj P;;;;sloj 37;;;;sloj 38;;;;sloj 39;;;;sloj 40;;;;meziloží slojí 37-38;;;;;;;meziloží slojí 38-39;;;;;;;meziloží slojí 39-40;;;;;;
;;;mocnost sloje P (cm);dobývaná mocnost sloje P (cm);kota sloje P (m n.m.);hloubka sloje P pod povrchem (m);mocnost sloje 37 (cm);dobývaná mocnost sloje 37 (cm);kota sloje 37 (m n.m.);hloubka sloje 37 pod povrchem (m);mocnost sloje 38 (cm);dobývaná mocnost sloje 38 (cm);kota sloje 38 (m n.m.);hloubka sloje 38 pod povrchem (m);mocnost sloje 39 (cm);dobývaná mocnost sloje 39 (cm);kota sloje 39 (m n.m.);hloubka sloje 39 pod povrchem (m);mocnost sloje 40;dobývaná mocnost sloje 40;kota sloje 40;hloubka sloje 40 pod povrchem;mocnost meziloží 37-38 (m);celková mocnost pískovců a slepenců (m);počet lavic pískovců a slepencůl o mocnosti větší než 5 m;počet lavic pískovců a slepencůl o mocnosti větší než 10 m;max mocnost lavic pískovců a slepenců (m);% kompetentních hornin;redukovaná pevnost hornin efektivního nadloží sloje 38 (Mpa);mocnost meziloží 38-39 (m);celková mocnost pískovců a slepenců (m);počet lavic pískovců a slepencůl o mocnosti větší než 5 m;počet lavic pískovců a slepencůl o mocnosti větší než 10 m;max mocnost lavic pískovců a slepenců (m);% kompetentních hornin;redukovaná pevnost hornin efektivního nadloží sloje 39 (Mpa);mocnost meziloží 39-40 (m);celková mocnost pískovců a slepenců (m);počet lavic pískovců a slepencůl o mocnosti větší než 5 m;počet lavic pískovců a slepencůl o mocnosti větší než 10 m;max mocnost lavic pískovců a slepenců (m);% kompetentních hornin;redukovaná pevnost hornin efektivního nadloží sloje 39 (Mpa)
C 37-92;-458106;-1103175,5;x;x;x;x;x;x;x;x;?;?;-393,1;643,1;557;557;-431,02;681,0;658;658;-488,75;738,8;x;x;x;x;x;x;x;32,35;23,32;1;1;15,67;72;x;51,02;48,32;3;1;18,97;95;x
"""

c36_images = glob.glob("vrty/C36_*.TIF")
c37_images = glob.glob("vrty/C37_*.TIF")

# --- Prepare few-shot content ---
few_shot_content = [
    {"type": "input_text", "text": f"Here is an example CSV for the following images: {', '.join(c36_images)}\n\n{c36_csv}"}
] + [
    {"type": "input_image", "image_url": image_to_data_url(img_path)}
    for img_path in c36_images
] + [
    {"type": "input_text", "text": f"Here is an example CSV for the following images: {', '.join(c37_images)}\n\n{c37_csv}"}
] + [
    {"type": "input_image", "image_url": image_to_data_url(img_path)}
    for img_path in c37_images
]

c38_images = glob.glob("vrty/C38_*.TIF")
new_image_messages = [
    {"type": "input_image", "image_url": image_to_data_url(img_path)}
    for img_path in c38_images
]

# --- Compose full prompt ---
user_input = [
    {
        "role": "user",
        "content": few_shot_content + [
            {"type": "input_text", "text": "Now, please produce the same CSV format for these new images:"}
        ] + new_image_messages
    }
]

/home/ondra/git/geo/phd/.venv/lib/python3.11/site-packages/PIL/Image.py:3451: DecompressionBombWarning: Image size (100763520 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
/home/ondra/git/geo/phd/.venv/lib/python3.11/site-packages/PIL/Image.py:3451: DecompressionBombWarning: Image size (100939520 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
/home/ondra/git/geo/phd/.venv/lib/python3.11/site-packages/PIL/Image.py:3451: DecompressionBombWarning: Image size (101150720 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


In [ ]:
from pprint import pprint
pprint(user_input)

In [7]:
response = client.responses.create(
    model="gpt-4.1",
    input=user_input
)

print(response.output_text)

Here is the extracted CSV based on the images provided (C38_1.TIF, C38_2.TIF, C38_3.TIF) following your required structure and filling in the values from the drill log sheets:

```csv
ID;x-coord;y-coord;sloj P;;;;sloj 37;;;;sloj 38;;;;sloj 39;;;;sloj 40;;;;meziloží slojí 37-38;;;;;;;meziloží slojí 38-39;;;;;;;meziloží slojí 39-40;;;;;;
;;;mocnost sloje P (cm);dobývaná mocnost sloje P (cm);kota sloje P (m n.m.);hloubka sloje P pod povrchem (m);mocnost sloje 37 (cm);dobývaná mocnost sloje 37 (cm);kota sloje 37 (m n.m.);hloubka sloje 37 pod povrchem (m);mocnost sloje 38 (cm);dobývaná mocnost sloje 38 (cm);kota sloje 38 (m n.m.);hloubka sloje 38 pod povrchem (m);mocnost sloje 39 (cm);dobývaná mocnost sloje 39 (cm);kota sloje 39 (m n.m.);hloubka sloje 39 pod povrchem (m);mocnost sloje 40;dobývaná mocnost sloje 40;kota sloje 40;hloubka sloje 40 pod povrchem;mocnost meziloží 37-38 (m);celková mocnost pískovců a slepenců (m);počet lavic pískovců a slepencůl o mocnosti větší než 5 m;počet lavic

In [8]:
ai_response = """
C 38-92;-457925,9;-1103254,3;x;x;x;x;x;x;x;x;550;550;-437,1;687,1;568;568;-495,8;745,8;660;660;-500,6;750,6;x;x;x;x;x;x;x;35,95;26,6;1;1;15,67;80;x;58,7;55;3;1;19,43;92;x
"""

In [9]:
should_be = """
C 38-92;-457925,9;-1103254,3;x;x;x;x;x;x;x;x;?;?;-397,45;647,5;550;550;-437,4;687,4;710;710;-495,8;745,8;x;x;x;x;x;x;x;34,45;27,4;2;1;20,3;80;x;51,3;47,65;4;1;17,1;93;x
"""

In [17]:
def diff_csv_strings(csv1, csv2):
    # Split into fields
    fields1 = csv1.split(";")
    fields2 = csv2.split(";")

    # Pad shorter list to avoid index errors
    max_len = max(len(fields1), len(fields2))
    fields1 += [""] * (max_len - len(fields1))
    fields2 += [""] * (max_len - len(fields2))

    # Compare field by field
    for i, (f1, f2) in enumerate(zip(fields1, fields2)):
        if f1 != f2:
            print(f"Column {i+1} ({number_to_excel_col(i)}):")
            print(f"  AI output : {f1}")
            print(f"  Expected  : {f2}")
            print()


def number_to_excel_col(n):
    """Convert 0-based number to Excel column name."""
    result = ""
    n += 1  # Shift to 1-based
    while n > 0:
        n, remainder = divmod(n - 1, 26)
        result = chr(65 + remainder) + result
    return result

In [19]:
diff_csv_strings(ai_response, should_be)

Column 12 (L):
  AI output : 550
  Expected  : ?

Column 13 (M):
  AI output : 550
  Expected  : ?

Column 14 (N):
  AI output : -437,1
  Expected  : -397,45

Column 15 (O):
  AI output : 687,1
  Expected  : 647,5

Column 16 (P):
  AI output : 568
  Expected  : 550

Column 17 (Q):
  AI output : 568
  Expected  : 550

Column 18 (R):
  AI output : -495,8
  Expected  : -437,4

Column 19 (S):
  AI output : 745,8
  Expected  : 687,4

Column 20 (T):
  AI output : 660
  Expected  : 710

Column 21 (U):
  AI output : 660
  Expected  : 710

Column 22 (V):
  AI output : -500,6
  Expected  : -495,8

Column 23 (W):
  AI output : 750,6
  Expected  : 745,8

Column 31 (AE):
  AI output : 35,95
  Expected  : 34,45

Column 32 (AF):
  AI output : 26,6
  Expected  : 27,4

Column 33 (AG):
  AI output : 1
  Expected  : 2

Column 35 (AI):
  AI output : 15,67
  Expected  : 20,3

Column 38 (AL):
  AI output : 58,7
  Expected  : 51,3

Column 39 (AM):
  AI output : 55
  Expected  : 47,65

Column 40 (AN):
  AI ou